In [8]:
# Cell 1: Import the necessary libraries
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import csv

# Also import os to manage file paths
import os

In [9]:
# Cell 2: Load your data from 'url_topic.csv'
csv_file_path = 'merged.csv'  # Path to your CSV file

# Load the data into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Print the initial data for reference
print("Initial Data:")
print(df.head())


Initial Data:
                                                 URL  \
0                           https://www.angi.comSLUG   
1                           https://www.angi.comSLUG   
2  https://www.angi.com/companylist/us/ks/frankfo...   
3  https://www.angi.com/companylist/us/nj/manning...   
4  https://www.angi.com/companylist/us/il/zeigler...   

                                               Topic  
0  I had to call him to set the appointment the d...  
1  We interviewed several people who claimed to h...  
2  There needs to be a zero stars in here. Horrib...  
3  I am a long standing customer of another appli...  
4  After reading his reviews I called Ben to see ...  


In [10]:
# Cell 3: Handling missing values and Vectorizing the text data using TfidfVectorizer
# Replace NaN values in the 'Topic' column with empty strings
df['Topic'].fillna('', inplace=True)

# Now apply the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Topic'])

# Optionally print the shape of the matrix for reference
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

/var/folders/1h/6t51kr7975n3sy_hp1x64fq00000gn/T/ipykernel_62961/2599455646.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Topic'].fillna('', inplace=True)


TF-IDF matrix shape: (100000, 92284)


In [11]:
# Cell 4: Dimensionality reduction using TruncatedSVD
svd = TruncatedSVD(n_components=100)  # Adjust n_components if needed
reduced_matrix = svd.fit_transform(tfidf_matrix)

# Optionally print the reduced matrix shape for reference
print(f"Reduced matrix shape: {reduced_matrix.shape}")

Reduced matrix shape: (100000, 100)


In [7]:
# Cell 5: Store the cosine similarity in a memmap file for memory-efficient access

# Create a memory-mapped file for the cosine similarity matrix
memmap_filename = 'cosine_similarity.dat'
shape = (reduced_matrix.shape[0], reduced_matrix.shape[0])  # Cosine similarity will be square matrix

# Create a memory-mapped array for storing the cosine similarity
cosine_sim_memmap = np.memmap(memmap_filename, dtype='float32', mode='w+', shape=shape)

# Calculate the cosine similarity in chunks and store it in the memory-mapped file
for i in range(reduced_matrix.shape[0]):
    cosine_sim_memmap[i] = cosine_similarity(reduced_matrix[i:i+1], reduced_matrix)[0]
    
# Flush changes to the file to ensure everything is written
cosine_sim_memmap.flush()

# Now you can access cosine_sim_memmap without keeping the entire matrix in memory
print("Cosine Similarity stored in memory-mapped file.")

Cosine Similarity stored in memory-mapped file.


In [ ]:
# Cell 6: Define functions to rank URLs and write the top related URLs to a CSV file
def rank_urls(index, cosine_sim_matrix):
    """
    Ranks URLs based on similarity scores.
    :param index: Index of the current URL
    :param cosine_sim_matrix: The full cosine similarity matrix (memory-mapped)
    :return: Sorted indices of similar URLs (excluding the current URL itself)
    """
    similarity_scores = cosine_sim_matrix[index]
    sorted_indices = np.argsort(-similarity_scores)
    return sorted_indices[1:]  # Exclude the current URL itself

def print_top_related_to_csv(urls, cosine_sim, top_n=5, filename="related_urls.csv"):
    """
    Writes the top related URLs to a CSV file.
    :param urls: List of URLs
    :param cosine_sim: Cosine similarity matrix (memory-mapped)
    :param top_n: Number of top related URLs to include
    :param filename: Output CSV filename
    """
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(["Source URL", "Target URL", "Score"])

        for index in range(len(urls)):
            ranked_indices = rank_urls(index, cosine_sim)
            for i in ranked_indices[:top_n]:
                similarity_score = cosine_sim[index][i]
                writer.writerow([urls[index], urls[i], f"{similarity_score:.3f}"])

# Now, let's generate the CSV file with the top related URLs
print_top_related_to_csv(df['URL'].tolist(), cosine_sim_memmap)